# تست سریع: استخراج بردار جابجایی واقعی هر Image سوپرسل از Phonopy

## هدف
قبل از ساخت Notebook 18 کامل، فقط می‌خواهیم تأیید کنیم که می‌توان از یک آبجکت `Phonopy`
(که از Notebook 16/17 می‌شناسیم) بردار جابجایی فیزیکی واقعی (`lattice displacement vector`,
به واحد Å یا fractional) هر یک از ۷۲ تصویر سوپرسل را استخراج کرد — تا بعداً به‌جای
`nn.Embedding(72, dim)` دلخواه (که در Notebook 17 باعث بدتر شدن MAE شد)، از این بردار
واقعی به‌عنوان feature ورودی استفاده کنیم.

## فرضیه‌ی این تست
هر اتم در سوپرسل (`phonon.supercell`) متعلق به یک "تصویر" (image) از سلول واحد است که
با یک بردار جابجایی شبکه‌ای `R = n1*a1 + n2*a2 + n3*a3` (با `n1,n2,n3` اعداد صحیح) از
سلول مرجع جدا شده. این بردار را می‌توان از طریق نگاشت سوپرسل به سلول واحد
(`s2u_map` / `supercell_to_unitcell_map`) و موقعیت کارتزی اتم‌ها به‌دست آورد.

اگر این تست موفق شود، یعنی برای هر یک از ۷۲ تصویر یک بردار سه‌بعدی `(dx, dy, dz)` ثابت
و معنادار داریم که می‌تواند جای embedding دلخواه Notebook 17 را بگیرد.

In [ ]:
!pip install -q phonopy -t /kaggle/working/custom_lib
import sys
sys.path.append('/kaggle/working/custom_lib')
print("✅ نصب شد")

In [ ]:
import numpy as np
import yaml
from pathlib import Path
from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.file_IO import parse_FORCE_CONSTANTS

FC_DIR     = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
BANDS_DIR  = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C'

FC_DIR_PATH = Path(FC_DIR)
BANDS_DIR_PATH = Path(BANDS_DIR)

fc_files = {f.stem: f for f in FC_DIR_PATH.glob('*.fc')}
band_yaml_files = {f.stem: f for f in BANDS_DIR_PATH.glob('*.yaml')}

common = sorted(set(fc_files) & set(band_yaml_files))
print(f"تعداد مواد یافت‌شده: {len(common)}")

# یک ماده‌ی نمونه برای تست انتخاب می‌کنیم
test_formula = common[0]
print(f"ماده‌ی تستی: {test_formula}")

## بازسازی آبجکت Phonopy برای ماده‌ی تستی (دقیقاً مثل Notebook 16/17)

برای این تست، چون فقط می‌خواهیم ساختار `supercell` را بررسی کنیم (نه دقت IFC)، یک
supercell ساده‌ی `3×3×1` (یا هر ابعادی که ۷۲/n_atoms تصویر بدهد) می‌سازیم. اگر این تست
موفق بود، در Notebook 18 از همان `supercell_dim` کشف‌شده‌ی واقعی هر ماده استفاده می‌کنیم.

In [ ]:
with open(band_yaml_files[test_formula]) as f:
    data = yaml.safe_load(f)

lattice = np.array(data['lattice'])
symbols = [p['symbol'] for p in data['points']]
frac_coords = np.array([p['coordinates'] for p in data['points']])
n_atoms = len(symbols)
print(f"n_atoms (سلول واحد): {n_atoms}")

unitcell = PhonopyAtoms(symbols=symbols, cell=lattice, scaled_positions=frac_coords)

fc = parse_FORCE_CONSTANTS(str(fc_files[test_formula]))
n_supercell = fc.shape[1]
n_images = n_supercell // n_atoms
print(f"n_supercell (کل): {n_supercell} -> n_images: {n_images}")

# فرض موقت: ابعاد سوپرسل را از روی n_images حدس می‌زنیم (در Notebook 18 واقعی از
# find_correct_supercell_dim استفاده می‌شود؛ اینجا فقط برای تست ساختاری کافی است)
# اگر n_images=9 باشد (مثل تجربه‌ی قبلی)، حدس اول 3x3x1 است
import math
def guess_dim(n):
    for a in range(1, n+1):
        if n % a == 0:
            rem = n // a
            for b in range(1, rem+1):
                if rem % b == 0:
                    c = rem // b
                    return (a, b, c)
    return (n, 1, 1)

dim_guess = guess_dim(n_images)
print(f"ابعاد سوپرسل (حدس اولیه برای تست): {dim_guess}")

phonon = Phonopy(unitcell, supercell_matrix=[[dim_guess[0],0,0],[0,dim_guess[1],0],[0,0,dim_guess[2]]])
phonon.force_constants = fc

print(f"✅ Phonopy ساخته شد. تعداد اتم سوپرسل: {len(phonon.supercell.symbols)}")

## استخراج بردار جابجایی هر Image (هسته‌ی اصلی این تست)

می‌خواهیم برای هر اتم در سوپرسل، بفهمیم:
1. به کدام اتم سلول واحد (unitcell) متعلق است (`s2u_map`)
2. بردار جابجایی شبکه‌ای آن نسبت به تصویر مرجع (image 0) چقدر است

سپس این بردارها را برای `n_images` تصویر یکتا (نه برای هر اتم، بلکه برای هر *تصویر* که
شامل چند اتم است) استخراج می‌کنیم.

In [ ]:
supercell = phonon.supercell
sc_positions_cart = supercell.positions  # موقعیت کارتزی اتم‌های سوپرسل (n_supercell, 3)
sc_symbols = supercell.symbols

# نگاشت سوپرسل به سلول واحد: هر اتم سوپرسل به کدام اندیس سلول واحد نگاشت می‌شود
s2u_map = supercell.s2u_map  # طول = n_supercell، مقدار = اندیس در سلول واحد (طول n_atoms تکرار شده)
print(f"s2u_map (10 مقدار اول): {s2u_map[:10]}")
print(f"طول s2u_map: {len(s2u_map)} (باید برابر n_supercell={n_supercell} باشد)")

unitcell_positions_cart = unitcell.positions  # (n_atoms, 3)

# برای هر اتم سوپرسل: بردار جابجایی = موقعیت سوپرسل - موقعیت اتم متناظرش در سلول واحد
displacement_vectors = []
for i in range(len(sc_positions_cart)):
    u_idx = s2u_map[i]
    disp = sc_positions_cart[i] - unitcell_positions_cart[u_idx]
    displacement_vectors.append(disp)

displacement_vectors = np.array(displacement_vectors)
print(f"\nشکل displacement_vectors: {displacement_vectors.shape} (باید (n_supercell, 3) باشد)")
print(f"چند نمونه‌ی اول:\n{displacement_vectors[:5]}")

## گروه‌بندی به ۷۲/n_atoms تصویر یکتا

هر "تصویر" شامل `n_atoms` اتم سوپرسل است که همگی یک بردار جابجایی یکسان دارند (چون یک
تصویر کامل از کل سلول واحد است که با یک بردار شبکه‌ای جابجا شده). پس باید ببینیم آیا
بردارهای جابجایی واقعاً در گروه‌های `n_atoms` تایی یکسان قرار می‌گیرند یا نه.

In [ ]:
# بررسی: آیا هر n_atoms اتم متوالی همان بردار جابجایی را دارند؟
# (ترتیب معمول phonopy: تمام اتم‌های image 0 اول، سپس image 1، و ...)
unique_displacements = []
for img_idx in range(n_images):
    start = img_idx * n_atoms
    end = start + n_atoms
    block = displacement_vectors[start:end]
    # چک کنیم آیا همه‌ی اتم‌های این بلوک یک بردار جابجایی یکسان دارند
    block_std = np.std(block, axis=0)
    is_consistent = np.allclose(block_std, 0, atol=1e-4)
    unique_displacements.append(block[0])
    print(f"Image {img_idx}: disp={block[0]}, consistent_within_block={is_consistent}")

unique_displacements = np.array(unique_displacements)
print(f"\n✅ شکل نهایی بردارهای جابجایی یکتا: {unique_displacements.shape} (باید ({n_images}, 3) باشد)")
print(f"\nتمام بردارهای جابجایی یکتا:\n{unique_displacements}")

# بررسی نهایی: آیا این بردارها متفاوت از هم هستند؟ (نباید تکراری باشند)
n_unique_actual = len(np.unique(unique_displacements.round(decimals=4), axis=0))
print(f"\nتعداد بردارهای واقعاً یکتا: {n_unique_actual} از {n_images}")

## نتیجه‌گیری این تست

اگر در خروجی بالا:
- `is_consistent=True` برای همه‌ی تصاویر باشد (یعنی هر بلوک از `n_atoms` اتم واقعاً یک
  بردار جابجایی مشترک دارد)
- و `n_unique_actual` برابر `n_images` باشد (یعنی همه‌ی بردارها واقعاً متفاوت‌اند، نه تکراری)

آن‌وقت این روش (`s2u_map` + تفاضل موقعیت) برای استخراج بردار جابجایی واقعی هر تصویر کار
می‌کند و می‌توانیم در Notebook 18 از این بردار (نرمال‌شده بر اساس طول لاتیس) به‌جای
`nn.Embedding` دلخواه استفاده کنیم.

⚠️ اگر `is_consistent=False` بود، یعنی ترتیب اتم‌های سوپرسل در Phonopy با فرض «همه‌ی
اتم‌های یک تصویر پشت‌سرهم» مطابقت ندارد و باید روش گروه‌بندی را اصلاح کنیم (مثلاً بر
اساس خودِ بردار جابجایی به‌جای فرض ترتیب).

**لطفاً خروجی کامل این نوت‌بوک (همه‌ی پرینت‌ها) را برای من بفرستید تا بر اساس آن Notebook 18 را بسازم.**